# CSV Creation — Document-Level Dataset

This notebook creates document-level CSV files from the Markdown corpus.

It reads:

- `markdown/cleaned/policy/`
- `markdown/cleaned/sentiment/`
- `markdown/synthetic/sentiment/mds/`

It writes:

- `csvs/raw/policy/policy_documents_full.csv`
- `csvs/raw/sentiment/sentiment_documents_full.csv`
- `csvs/raw/synthetic/sentiment/synthetic_sentiment_documents_full.csv`
- `csvs/raw/documents_full_all.csv`
- `csvs/raw/documents_full_summary.csv`

For now, there is **no chunking**.  
Each Markdown document becomes one row in the CSV.

In [1]:
from pathlib import Path
import re
import pandas as pd
from datetime import datetime

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """
    Find the project root by walking upward until a folder containing
    both markdown/ and csvs/ is found.
    """
    if start is None:
        start = Path.cwd()

    start = start.resolve()

    for candidate in [start] + list(start.parents):
        if (candidate / "markdown").exists() and (candidate / "csvs").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root."
    )


PROJECT_ROOT = find_project_root()

print("Project root:", PROJECT_ROOT)

Project root: /home/nsirim/Github/mscdsa/msc


In [3]:
# Input folders
POLICY_MD_DIR = PROJECT_ROOT / "markdown" / "cleaned" / "policy"
SENTIMENT_MD_DIR = PROJECT_ROOT / "markdown" / "cleaned" / "sentiment"
SYNTHETIC_SENTIMENT_MD_DIR = PROJECT_ROOT / "markdown" / "synthetic" / "sentiment" / "mds"

# Output folders
CSVS_NEW_DIR = PROJECT_ROOT / "csvs" / "raw"
POLICY_OUT_DIR = CSVS_NEW_DIR / "policy"
SENTIMENT_OUT_DIR = CSVS_NEW_DIR / "sentiment"
SYNTHETIC_SENTIMENT_OUT_DIR = CSVS_NEW_DIR / "synthetic" / "sentiment"

for folder in [
    CSVS_NEW_DIR,
    POLICY_OUT_DIR,
    SENTIMENT_OUT_DIR,
    SYNTHETIC_SENTIMENT_OUT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Policy Markdown folder:", POLICY_MD_DIR)
print("Original sentiment Markdown folder:", SENTIMENT_MD_DIR)
print("Synthetic sentiment Markdown folder:", SYNTHETIC_SENTIMENT_MD_DIR)
print("Output folder:", CSVS_NEW_DIR)

Policy Markdown folder: /home/nsirim/Github/mscdsa/msc/markdown/cleaned/policy
Original sentiment Markdown folder: /home/nsirim/Github/mscdsa/msc/markdown/cleaned/sentiment
Synthetic sentiment Markdown folder: /home/nsirim/Github/mscdsa/msc/markdown/synthetic/sentiment/mds
Output folder: /home/nsirim/Github/mscdsa/msc/csvs/raw


In [4]:
def list_markdown_files(folder: Path) -> list[Path]:
    """
    List Markdown files recursively.
    Hidden/system files are ignored.
    """
    if not folder.exists():
        print(f"Warning: folder does not exist: {folder}")
        return []

    files = sorted(folder.rglob("*.md"))

    files = [
        path for path in files
        if not any(part.startswith(".") for part in path.parts)
    ]

    return files


policy_files = list_markdown_files(POLICY_MD_DIR)
sentiment_files = list_markdown_files(SENTIMENT_MD_DIR)
synthetic_sentiment_files = list_markdown_files(SYNTHETIC_SENTIMENT_MD_DIR)

print("Policy files:", len(policy_files))
print("Original sentiment files:", len(sentiment_files))
print("Synthetic sentiment files:", len(synthetic_sentiment_files))

print("\nExamples:")
for path in policy_files[:3]:
    print("POLICY:", path.relative_to(PROJECT_ROOT))

for path in sentiment_files[:3]:
    print("SENTIMENT:", path.relative_to(PROJECT_ROOT))

for path in synthetic_sentiment_files[:3]:
    print("SYNTHETIC:", path.relative_to(PROJECT_ROOT))

Policy files: 56
Original sentiment files: 17
Synthetic sentiment files: 53

Examples:
POLICY: markdown/cleaned/policy/AI competency framework  for students.md
POLICY: markdown/cleaned/policy/AI competency framework  for teachers.md
POLICY: markdown/cleaned/policy/australia/AHISA MEMBER SURVEY REPORT Gen AI in Australian independent schools.md
SENTIMENT: markdown/cleaned/sentiment/051024_Insights into AI and the youth sector.md
SENTIMENT: markdown/cleaned/sentiment/20251119-Les-Francais-lIA.md
SENTIMENT: markdown/cleaned/sentiment/3RemittancesReview-Volume8-Issue4Saajdmalakand.md
SYNTHETIC: markdown/synthetic/sentiment/mds/sentiment_press_release_aa.md
SYNTHETIC: markdown/synthetic/sentiment/mds/sentiment_press_release_ab.md
SYNTHETIC: markdown/synthetic/sentiment/mds/sentiment_press_release_ac.md


In [5]:
def normalize_document_text(text: str) -> str:
    """
    Light normalization for document-level CSV.

    This does not chunk the document and does not remove meaningful content.
    It only removes remaining technical Markdown artifacts if they exist.
    """
    # Remove HTML comments such as image metadata if any remain.
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # Remove Markdown image links if any remain.
    text = re.sub(r"\\?!\[[^\]]*\]\([^)]+\)", " ", text)

    # Remove raw local image paths if any remain.
    text = re.sub(
        r"/home/[^\s)]+?\.(png|jpg|jpeg|webp)",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove code fences but keep any inner text.
    text = re.sub(r"```[\w-]*", " ", text)

    # Normalize non-breaking spaces.
    text = text.replace("\xa0", " ")

    # Normalize line endings.
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Collapse spaces inside each line.
    lines = [
        re.sub(r"[ \t]+", " ", line).strip()
        for line in text.split("\n")
    ]

    # Remove excessive blank lines.
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text))


def infer_country_from_policy_path(path: Path) -> str:
    """
    Infer country from:
    markdown/cleaned/policy/<country>/file.md
    """
    try:
        rel = path.relative_to(POLICY_MD_DIR)

        if len(rel.parts) >= 2:
            return rel.parts[0]

    except ValueError:
        pass

    return ""


def infer_synthetic_type(filename: str) -> str:
    """
    Infer synthetic type from filenames such as:

    sentiment_comment_001.md
    sentiment_institutional_statement_001.md
    sentiment_report_summary_001.md
    """
    name = filename.lower()

    if "institutional_statement" in name:
        return "institutional_statement"

    if "report_summary" in name:
        return "report_summary"

    if "comment" in name:
        return "comment"

    return ""


def make_doc_id(prefix: str, path: Path) -> str:
    """
    Create a stable document id from prefix and file stem.
    """
    stem = re.sub(r"[^a-zA-Z0-9]+", "_", path.stem)
    stem = stem.strip("_").lower()

    return f"{prefix}_{stem}"

In [6]:
def build_records_for_files(
    files: list[Path],
    *,
    corpus: str,
    source_type: str,
    base_dir: Path,
    doc_prefix: str,
) -> list[dict]:
    records = []

    for path in files:
        raw_text = path.read_text(encoding="utf-8", errors="replace")
        text = normalize_document_text(raw_text)

        if not text:
            print(f"Warning: empty text after normalization: {path.relative_to(PROJECT_ROOT)}")
            continue

        country = ""
        synthetic_type = ""

        if corpus == "policy":
            country = infer_country_from_policy_path(path)

        if source_type == "synthetic":
            country = "synthetic_sentiment"
            synthetic_type = infer_synthetic_type(path.name)

        try:
            source_file = str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            source_file = str(path)

        try:
            relative_source_file = str(path.relative_to(base_dir))
        except ValueError:
            relative_source_file = path.name

        records.append(
            {
                "doc_id": make_doc_id(doc_prefix, path),
                "filename": path.name,
                "source_file": source_file,
                "relative_source_file": relative_source_file,
                "corpus": corpus,
                "source_type": source_type,
                "synthetic_type": synthetic_type,
                "country": country,
                "text": text,
                "char_count": len(text),
                "word_count": count_words(text),
            }
        )

    return records


policy_records = build_records_for_files(
    policy_files,
    corpus="policy",
    source_type="original",
    base_dir=POLICY_MD_DIR,
    doc_prefix="policy",
)

sentiment_records = build_records_for_files(
    sentiment_files,
    corpus="sentiment",
    source_type="original",
    base_dir=SENTIMENT_MD_DIR,
    doc_prefix="sentiment_original",
)

synthetic_sentiment_records = build_records_for_files(
    synthetic_sentiment_files,
    corpus="sentiment",
    source_type="synthetic",
    base_dir=SYNTHETIC_SENTIMENT_MD_DIR,
    doc_prefix="sentiment_synthetic",
)

print("Policy records:", len(policy_records))
print("Original sentiment records:", len(sentiment_records))
print("Synthetic sentiment records:", len(synthetic_sentiment_records))

Policy records: 56
Original sentiment records: 17
Synthetic sentiment records: 53


In [7]:
df_policy = pd.DataFrame(policy_records)
df_sentiment = pd.DataFrame(sentiment_records)
df_synthetic_sentiment = pd.DataFrame(synthetic_sentiment_records)

df_all = pd.concat(
    [
        df_policy,
        df_sentiment,
        df_synthetic_sentiment,
    ],
    ignore_index=True,
)

print("Policy dataframe:", df_policy.shape)
print("Original sentiment dataframe:", df_sentiment.shape)
print("Synthetic sentiment dataframe:", df_synthetic_sentiment.shape)
print("All dataframe:", df_all.shape)

df_all.head()

Policy dataframe: (56, 11)
Original sentiment dataframe: (17, 11)
Synthetic sentiment dataframe: (53, 11)
All dataframe: (126, 11)


,doc_id,filename,source_file,relative_source_file,corpus,source_type,synthetic_type,country,text,char_count,word_count
0,policy_ai_competency_framework_for_students,AI competency framework for students.md,markdown/cleaned/policy/AI competency framewor...,AI competency framework for students.md,policy,original,,,# AI competency framework\n\n## for students\n...,287716,31077
1,policy_ai_competency_framework_for_teachers,AI competency framework for teachers.md,markdown/cleaned/policy/AI competency framewor...,AI competency framework for teachers.md,policy,original,,,# AI competency framework for teachers\n\n## U...,212558,23253
2,policy_ahisa_member_survey_report_gen_ai_in_au...,AHISA MEMBER SURVEY REPORT Gen AI in Australia...,markdown/cleaned/policy/australia/AHISA MEMBER...,australia/AHISA MEMBER SURVEY REPORT Gen AI in...,policy,original,,australia,AHISA MEMBER SURVEY\n\n# The use of generative...,56285,8503
3,policy_ai_systems_in_teaching_and_learning_pri...,AI systems in teaching and learning_ Principle...,markdown/cleaned/policy/australia/AI systems i...,australia/AI systems in teaching and learning_...,policy,original,,australia,# AI systems in teaching and learning: princip...,82387,11037
4,policy_asc_artificial_intelligence_policy_2025,ASC-Artificial-Intelligence-Policy-2025.md,markdown/cleaned/policy/australia/ASC-Artifici...,australia/ASC-Artificial-Intelligence-Policy-2...,policy,original,,australia,# Artificial Intelligence (AI) Policy\n\n## Pu...,10353,1528


In [8]:
policy_csv = POLICY_OUT_DIR / "policy_documents_full.csv"
sentiment_csv = SENTIMENT_OUT_DIR / "sentiment_documents_full.csv"
synthetic_sentiment_csv = SYNTHETIC_SENTIMENT_OUT_DIR / "synthetic_sentiment_documents_full.csv"
all_csv = CSVS_NEW_DIR / "documents_full_all.csv"

df_policy.to_csv(policy_csv, index=False, encoding="utf-8")
df_sentiment.to_csv(sentiment_csv, index=False, encoding="utf-8")
df_synthetic_sentiment.to_csv(synthetic_sentiment_csv, index=False, encoding="utf-8")
df_all.to_csv(all_csv, index=False, encoding="utf-8")

print("Saved:")
print("-", policy_csv.relative_to(PROJECT_ROOT))
print("-", sentiment_csv.relative_to(PROJECT_ROOT))
print("-", synthetic_sentiment_csv.relative_to(PROJECT_ROOT))
print("-", all_csv.relative_to(PROJECT_ROOT))

Saved:
- csvs/raw/policy/policy_documents_full.csv
- csvs/raw/sentiment/sentiment_documents_full.csv
- csvs/raw/synthetic/sentiment/synthetic_sentiment_documents_full.csv
- csvs/raw/documents_full_all.csv


In [9]:
def summarize_dataframe(df: pd.DataFrame, name: str):
    print(f"\n{name}")
    print("-" * len(name))

    print("Documents:", len(df))

    if len(df) == 0:
        print("Total characters: 0")
        print("Total words: 0")
        return

    print("Total characters:", int(df["char_count"].sum()))
    print("Total words:", int(df["word_count"].sum()))
    print("Average characters per document:", round(df["char_count"].mean(), 2))
    print("Average words per document:", round(df["word_count"].mean(), 2))


summarize_dataframe(df_policy, "Policy")
summarize_dataframe(df_sentiment, "Original sentiment")
summarize_dataframe(df_synthetic_sentiment, "Synthetic sentiment")
summarize_dataframe(df_all, "All documents")


Policy
------
Documents: 56
Total characters: 3306541
Total words: 478997
Average characters per document: 59045.38
Average words per document: 8553.52

Original sentiment
------------------
Documents: 17
Total characters: 842718
Total words: 129535
Average characters per document: 49571.65
Average words per document: 7619.71

Synthetic sentiment
-------------------
Documents: 53
Total characters: 385364
Total words: 57589
Average characters per document: 7271.02
Average words per document: 1086.58

All documents
-------------
Documents: 126
Total characters: 4534623
Total words: 666121
Average characters per document: 35989.07
Average words per document: 5286.67


In [10]:
summary_by_source = (
    df_all
    .groupby(
        [
            "corpus",
            "source_type",
            "synthetic_type",
        ],
        dropna=False,
    )
    .agg(
        documents=("doc_id", "count"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
        avg_characters=("char_count", "mean"),
        avg_words=("word_count", "mean"),
    )
    .reset_index()
)

summary_by_source

,corpus,source_type,synthetic_type,documents,total_characters,total_words,avg_characters,avg_words
0,policy,original,,56,3306541,478997,59045.375000,8553.517857
1,sentiment,original,,17,842718,129535,49571.647059,7619.705882
2,sentiment,synthetic,,53,385364,57589,7271.018868,1086.584906


In [11]:
summary_csv = CSVS_NEW_DIR / "documents_full_summary.csv"
summary_by_source.to_csv(summary_csv, index=False, encoding="utf-8")

print("Saved summary:")
print(summary_csv.relative_to(PROJECT_ROOT))

Saved summary:
csvs/raw/documents_full_summary.csv


In [12]:
balance = (
    df_all
    .groupby(
        [
            "corpus",
            "source_type",
        ],
        dropna=False,
    )
    .agg(
        documents=("doc_id", "count"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
    )
    .reset_index()
)

balance

,corpus,source_type,documents,total_characters,total_words
0,policy,original,56,3306541,478997
1,sentiment,original,17,842718,129535
2,sentiment,synthetic,53,385364,57589


In [13]:
policy_chars = df_policy["char_count"].sum() if len(df_policy) else 0
original_sentiment_chars = df_sentiment["char_count"].sum() if len(df_sentiment) else 0
synthetic_sentiment_chars = df_synthetic_sentiment["char_count"].sum() if len(df_synthetic_sentiment) else 0
combined_sentiment_chars = original_sentiment_chars + synthetic_sentiment_chars

print("Policy characters:", policy_chars)
print("Original sentiment characters:", original_sentiment_chars)
print("Synthetic sentiment characters:", synthetic_sentiment_chars)
print("Combined sentiment characters:", combined_sentiment_chars)
print("Difference policy - combined sentiment:", policy_chars - combined_sentiment_chars)

Policy characters: 3306541
Original sentiment characters: 842718
Synthetic sentiment characters: 385364
Combined sentiment characters: 1228082
Difference policy - combined sentiment: 2078459


In [14]:
preview_columns = [
    "doc_id",
    "filename",
    "corpus",
    "source_type",
    "synthetic_type",
    "country",
    "char_count",
    "word_count",
]

df_all[preview_columns].head(20)

,doc_id,filename,corpus,source_type,synthetic_type,country,char_count,word_count
0,policy_ai_competency_framework_for_students,AI competency framework for students.md,policy,original,,,287716,31077
1,policy_ai_competency_framework_for_teachers,AI competency framework for teachers.md,policy,original,,,212558,23253
2,policy_ahisa_member_survey_report_gen_ai_in_au...,AHISA MEMBER SURVEY REPORT Gen AI in Australia...,policy,original,,australia,56285,8503
3,policy_ai_systems_in_teaching_and_learning_pri...,AI systems in teaching and learning_ Principle...,policy,original,,australia,82387,11037
4,policy_asc_artificial_intelligence_policy_2025,ASC-Artificial-Intelligence-Policy-2025.md,policy,original,,australia,10353,1528
5,policy_artificial_intelligence_policy,Artificial Intelligence Policy.md,policy,original,,australia,18391,2640
6,policy_australian_framework_for_generative_ai_...,Australian Framework for Generative AI in Scho...,policy,original,,australia,14208,2022
7,policy_doai_24_educator_guide,DOAI-24-Educator-Guide.md,policy,original,,australia,13487,2069
8,policy_generative_ai_usage_policy,Generative_AI_Usage_Policy.md,policy,original,,australia,16137,2184
9,policy_generative_ai_in_the_australian_educati...,Generative_AI_in_the_Australian_education_syst...,policy,original,,australia,93346,13865


In [15]:
duplicates = df_all[df_all.duplicated("doc_id", keep=False)]

print("Duplicated doc_id rows:", len(duplicates))

if len(duplicates):
    display(
        duplicates[
            [
                "doc_id",
                "source_file",
            ]
        ]
        .sort_values("doc_id")
        .head(30)
    )

Duplicated doc_id rows: 0


In [16]:
small_docs = df_all.sort_values("word_count").head(20)

small_docs[
    [
        "doc_id",
        "filename",
        "corpus",
        "source_type",
        "synthetic_type",
        "word_count",
        "char_count",
        "source_file",
    ]
]

,doc_id,filename,corpus,source_type,synthetic_type,word_count,char_count,source_file
125,sentiment_synthetic_sentiment_public_sentiment_ba,sentiment_public_sentiment_ba.md,sentiment,synthetic,,183,1244,markdown/synthetic/sentiment/mds/sentiment_pub...
11,policy_review_of_the_australian_framework_for_...,Review of the Australian Framework for GenAI i...,policy,original,,505,3543,markdown/cleaned/policy/australia/Review of th...
13,policy_use_of_generative_ai_in_decyp_schools_p...,Use-of-Generative-AI-in-DECYP-Schools-Policy.md,policy,original,,536,3685,markdown/cleaned/policy/australia/Use-of-Gener...
26,policy_2025_11_18_opening_statement_dr_dan_mcq...,2025-11-18_opening-statement-dr-dan-mcquillan-...,policy,original,,694,4166,markdown/cleaned/policy/ireland/2025-11-18_ope...
18,policy_generative_artificial_intelligence_in_s...,generative-artificial-intelligence-in-schools-...,policy,original,,718,5399,markdown/cleaned/policy/australia/generative-a...
29,policy_2026_01_20_opening_statement_graham_lon...,2026-01-20_opening-statement-graham-long-chief...,policy,original,,776,4832,markdown/cleaned/policy/ireland/2026-01-20_ope...
30,policy_2026_01_20_opening_statement_ruth_kenne...,2026-01-20_opening-statement-ruth-kennedy-reve...,policy,original,,820,5210,markdown/cleaned/policy/ireland/2026-01-20_ope...
20,policy_ai_policy_and_regulations_of_france_com...,AI Policy and Regulations of France: Compreh...,policy,original,,890,5588,markdown/cleaned/policy/france/AI Policy and ...
82,sentiment_synthetic_sentiment_press_release_aj,sentiment_press_release_aj.md,sentiment,synthetic,,1001,7290,markdown/synthetic/sentiment/mds/sentiment_pre...
91,sentiment_synthetic_sentiment_press_release_as,sentiment_press_release_as.md,sentiment,synthetic,,1010,7306,markdown/synthetic/sentiment/mds/sentiment_pre...


In [17]:
large_docs = df_all.sort_values("char_count", ascending=False).head(20)

large_docs[
    [
        "doc_id",
        "filename",
        "corpus",
        "source_type",
        "synthetic_type",
        "word_count",
        "char_count",
        "source_file",
    ]
]

,doc_id,filename,corpus,source_type,synthetic_type,word_count,char_count,source_file
0,policy_ai_competency_framework_for_students,AI competency framework for students.md,policy,original,,31077,287716,markdown/cleaned/policy/AI competency framewor...
21,policy_l_intelligence_artificielle_dans_les_ta...,L’intelligence artificielle dans les établiss...,policy,original,,39136,243706,markdown/cleaned/policy/france/L’intelligence ...
1,policy_ai_competency_framework_for_teachers,AI competency framework for teachers.md,policy,original,,23253,212558,markdown/cleaned/policy/AI competency framewor...
35,policy_oco_ai_policy_spotlight_report,OCO-AI-Policy-Spotlight-report-.md,policy,original,,32882,209937,markdown/cleaned/policy/ireland/OCO-AI-Policy-...
54,policy_ai_report_us_gov,ai_report_us_gov.md,policy,original,,31430,204088,markdown/cleaned/policy/usa/ai_report_us_gov.md
60,sentiment_original_ai4t_wp3_d3_3_nr_ireland,AI4T_WP3_D3.3_NR_Ireland.md,sentiment,original,,28573,184685,markdown/cleaned/sentiment/AI4T_WP3_D3.3_NR_Ir...
39,policy_digital_strategy_for_schools_to_2027,digital-strategy-for-schools-to-2027.md,policy,original,,28913,173379,markdown/cleaned/policy/ireland/digital-strate...
48,policy_guidetoaiinschools,GuideToAIInSchools.md,policy,original,,25573,151570,markdown/cleaned/policy/usa/GuideToAIInSchools.md
15,policy_ai_in_australian_education_snapshot_pri...,ai-in-australian-education-snapshot-principles...,policy,original,,21712,147759,markdown/cleaned/policy/australia/ai-in-austra...
14,policy_ai_cognitive_offloading_and_implication...,ai-cognitive-offloading-and-implications-for-e...,policy,original,,21630,146705,markdown/cleaned/policy/australia/ai-cognitive...
